In [5]:
import torchxrayvision as xrv


In [17]:
model.pathologies

['Atelectasis',
 'Consolidation',
 '',
 'Pneumothorax',
 'Edema',
 '',
 '',
 'Effusion',
 'Pneumonia',
 '',
 'Cardiomegaly',
 '',
 '',
 '',
 'Lung Lesion',
 'Fracture',
 'Lung Opacity',
 'Enlarged Cardiomediastinum']

In [18]:
dataset.pathologies

['Atelectasis',
 'Cardiomegaly',
 'Consolidation',
 'Edema',
 'Enlarged Cardiomediastinum',
 'Fracture',
 'Lung Lesion',
 'Lung Opacity',
 'Effusion',
 'Pleural Other',
 'Pneumonia',
 'Pneumothorax',
 'Support Devices']

In [21]:
np.where(item['lab'] == 1)[0]

array([], dtype=int64)

In [29]:
torch.nn.Sigmoid()(torch.tensor(outputs))

tensor([0.6991, 0.6503, 0.5000, 0.6239, 0.6519, 0.5000, 0.5000, 0.6841, 0.6449,
        0.5000, 0.6083, 0.5000, 0.5000, 0.5000, 0.6353, 0.6386, 0.7037, 0.6244],
       dtype=torch.float64)

In [35]:
import numpy as np
import torch
import torchvision
from sklearn.metrics import roc_auc_score
# dataset = xrv.datasets.NIH_Dataset('/home/royhirsch/datasets/chest_x_14/images-224/images-224')
# model = xrv.models.DenseNet(weights="densenet121-res224-nih") # NIH chest X-ray8

dataset = xrv.datasets.CheX_Dataset('/home/royhirsch/datasets/chexpert.zip/chexpertchestxrays-u20210408/CheXpert-v1.0')
model = xrv.models.DenseNet(weights="densenet121-res224-chex") # CheXpert (Stanford)
# model.eval()
# model = xrv.models.DenseNet(weights="densenet121-res224-mimic_nb") # MIMIC-CXR (MIT)
# model = xrv.models.DenseNet(weights="densenet121-res224-mimic_ch") # MIMIC-CXR (MIT)

transform = torchvision.transforms.Compose([xrv.datasets.XRayCenterCrop(),
                                            xrv.datasets.XRayResizer(224)])
aucs = []
acc = []
for i in range(50):
    try:
        item = dataset.__getitem__(i)
    except:
        pass

    labels = np.where(item['lab'] == 1)[0]
    if not len(labels):
        pass
    elif len(labels) > 1:
        pass
    else:
        if dataset.pathologies[labels[0]] in model.pathologies:
            img = item['img']
            img = transform(img)
            img = torch.from_numpy(img)
            with torch.no_grad():
                outputs = model(img[None,...]) 
            outputs = outputs[0, :].detach().numpy() * np.array([1 if len(i) else 0 for i in model.pathologies])
            one_hot = np.zeros(len(dataset.pathologies))
            one_hot[labels[0]] = 1
            aucs.append(roc_auc_score(one_hot, outputs))
            print(model.pathologies[np.argmax(outputs)], ' | ',
                dataset.pathologies[labels[0]])
            acc.append(model.pathologies[np.argmax(outputs)] ==  dataset.pathologies[labels[0]])
print(np.mean(acc))
print(np.mean(aucs))


ValueError: Found input variables with inconsistent numbers of samples: [13, 18]